In [59]:
import numpy as np 
import pandas as pd
import re 
import datetime
import traceback as tb
import sys
import math
import unicodedata
from datetime import time

In [60]:
df_ctd = pd.read_excel(r'C:\Users\felipe.abarzua\Desktop\workspace\DATA_AMBIENTALES\Proyecto Seguimiento Ambiental\CTD_2019-2024\6_BD_CTD_SEGUIMIENTO_2024.xlsx')

In [62]:
df_ctd

,Nombre del Proyecto:,"ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO, 2024 -2025",Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29
0,Codigo del Proyecto:,656-171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Etapa Proyecto:,Objetivo 1. Actividad 2. Muestreos y Análisis ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Jefe Proyecto,Johana Ojeda Palma,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Institución:,Instituto de Fomento Pesquero,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sectores:,"Estuario de Reloncaví, Seno de Reloncaví, Golf...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6571,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,247,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6572,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,248,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6573,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,249,251,...,NaN,NaN,NaN,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6
6574,ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULT...,656-171,2024-07-25 00:00:00,2024-07-25 00:00:00,CTD SeaBird SBE 19plus SERIAL NO. 01908197//...,-45.668,-73.299,32,250,251,...,11.8915,1.8955,19.188,NaN,XI,NaN,32,FIORDO QUITRALCO,13:07:00,6


In [ ]:
def limpieza_ctd(df):

    # Creamos una copia del dataset
    df_ctd_copy = df.copy()

    # Eliminamos las filas innecesarias
    df_ctd_copy = df_ctd_copy.iloc[22:  , : ]

    # Dejamos la primera fila como columnas
    df_ctd_copy.columns = df_ctd_copy.iloc[0]

    # Eliminamos la primera fila 
    df_ctd_copy = df_ctd_copy.iloc[1: , :]

    #Reiniciamos los indices
    df_ctd_copy = df_ctd_copy.reset_index(drop = True)

    #Agregamos la columna ID 
    df_ctd_copy.insert(0 , 'ID' , df_ctd_copy.index + 1)

    # Modificamos las columnas de estación ya que existen dos de ellas
    df_ctd_copy.columns.values[8] = 'ESTACION_1'
    df_ctd_copy.columns.values[27] = 'ESTACION_2'

    #Eliminamos la estacion_2 ya que es innecesaria
    df_ctd_copy.drop(columns=['ESTACION_2'], axis=1 , inplace = True)

    # Renombramos la ESTACION_1 por ESTACION
    df_ctd_copy.columns.values[8] = 'ESTACION'

    #Creamos la lista de columnas 
    columnas = df_ctd_copy.columns.to_list()
    

    #Creamos las columnas fijas y variables
    columnas_fijas = ['ID', 'NOMB_PROY', 'COD_PROY', 'FECHA_INI', 'FECHA_TER', 'EQUIPO', 'LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT' , 'HORA_INICIO' , 'COD_REG' , 'SECTOR']
    columnas_variables = [ 'TEMPERATUR', 'OXIG_ml/L', 'OXIG_mg/L', 'OXIG_%sat', 'OX_umol/kg','SALINIDAD', 'DENSIDAD', 'CLOROFILA', 'FEOPIGMEN' , 'D_SECCHI']

    
    #Realizamos un melt para convertir las columnas_variables en filas 
    df_ctd_copy = pd.melt(df_ctd_copy , id_vars= columnas_fijas , value_vars= columnas_variables , var_name= 'VARIABLE' , value_name='VALOR')


        #Vamos a reemplazar los valores e HORA_INICIO ya que están mal escritos

    def normalizar_hora_string(hora_str):

            #Normaliza una cadena de tiempo para asegurar que los minutos tengan dos dígitos.
            #Ej: '12:5' se convierte en '12:05'.
            
        if pd.isna(hora_str): # Maneja posibles valores NaN/nulos si los hubiera
            return hora_str

        partes = str(hora_str).split(':')
        if len(partes) != 2:
                # Manejar casos donde el formato no es 'HH:MM' (ej. ya está mal, o es un dato inesperado)
                # Puedes decidir si quieres levantar un error, devolver el original, o un valor específico.
                # Por simplicidad, devolveremos el original si el formato no es el esperado de dos partes.
            return hora_str

        horas = partes[0]
        minutos = partes[1]

        if len(minutos) == 1:
            minutos = '0' + minutos # Añadir el cero delante si es un solo dígito

            return f"{horas}:{minutos}"
        
        #Aplicamos la funcion de hora a la columnas HORA_INICIO
        df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(normalizar_hora_string)

        # Creamos una funcion para convertir la columna de HORA_INICIO a datetime para luego Crear una columna de fecha y hora 
    def convertir_hora(time_value):
        if isinstance(time_value, str): 
                # Separamos el ":" del texto y lo convertimos a entero y obtenemos dos variables
                horas, minutos = map(int, time_value.split(':'))
                #Retornamos los valores de Horas y minutos
                return datetime.time(horas, minutos)
        else:
                # En caso que ya es datetime.time se deja como esta
            return time_value


    df_ctd_copy['HORA_INICIO'] = df_ctd_copy['HORA_INICIO'].apply(convertir_hora)


         #Cambiamos los tipos de FECHA_INI y FECHA_FIN a datetime

    df_ctd_copy['FECHA_INI'] = pd.to_datetime(df_ctd_copy['FECHA_INI'])
    df_ctd_copy['FECHA_TER'] = pd.to_datetime(df_ctd_copy['FECHA_TER'])

        #FECHA INI CONVERTIDA A STRING 
    df_ctd_copy['FECHA_INI_STR'] =  df_ctd_copy['FECHA_INI'].dt.strftime("%d/%m/%Y")
    df_ctd_copy['FECHA_TERMINO'] = df_ctd_copy['FECHA_TER'].dt.strftime("%d/%m/%Y")

        
        #Creamos una funcion lambda para combinar FECHA_INI con la HORA_INI
    df_ctd_copy['FECHA_INICIO'] = df_ctd_copy.apply(
        lambda row: datetime.datetime.strptime(row['FECHA_INI_STR'], "%d/%m/%Y").replace(
            hour=row['HORA_INICIO'].hour,
            minute=row['HORA_INICIO'].minute,
            second=row['HORA_INICIO'].second
            ),
            axis=1
        )

    # Modificamos el formato de FECHA_INICIO 
    df_ctd_copy['FECHA_INICIO'] = df_ctd_copy['FECHA_INICIO'].dt.strftime("%d/%m/%Y %H:%M:%S")

    # Eliminamos las columnas innecesarias

    df_ctd_copy = df_ctd_copy[['ID', 'NOMB_PROY', 'COD_PROY', 'EQUIPO','LATITUD', 'LONGUITUD', 'ESTACION', 'PROF_EQ', 'PROF_SECT', 'COD_REG', 'SECTOR', 'VARIABLE', 'VALOR',
                                'FECHA_TERMINO', 'FECHA_INICIO']]

    # Cambiamos las variables
    df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'D_SECCHI' , 'DISCO SECCHI' , df_ctd_copy['VARIABLE'] )
    df_ctd_copy['VARIABLE'] = np.where(df_ctd_copy['VARIABLE'] == 'TEMPERATUR' , 'TEMPERATURA' , df_ctd_copy['VARIABLE'] )
        
     # Dejamos las variable en mayuscula
    df_ctd_copy['VARIABLE'] = df_ctd_copy['VARIABLE'].str.upper()

    #Renombramos las columnas

    df_ctd_copy = df_ctd_copy.rename(columns={'LONGUITUD' : 'LONGITUD' , 'PROF_EQ' : 'PROF_EQU'})

    #Corregimos los NOMB_PROY
    df_ctd_copy['NOMB_PROY'] = 'ESTUDIO DEL DESEMPEÑO AMBIENTAL DE LA ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO 2024-2025'

    #Renombramos los EQUIPO
    df_ctd_copy['EQUIPO'] = 'CTD SeaBird SBE 19plus SERIAL NO.01908197'

    #Eliminamos los valores incorrectos

    df_ctd_copy['VALOR'] = np.where(df_ctd_copy['VALOR'] == 'sd' , np.nan , df_ctd_copy['VALOR'])


    return df_ctd_copy , columnas



In [85]:
df_ctd_copy , columnas = limpieza_ctd(df_ctd)

In [70]:
df_ctd_copy.columns

Index(['ID', 'NOMB_PROY', 'COD_PROY', 'EQUIPO', 'LATITUD', 'LONGITUD',
       'ESTACION', 'PROF_EQU', 'PROF_SECT', 'COD_REG', 'SECTOR', 'VARIABLE',
       'VALOR', 'FECHA_TERMINO', 'FECHA_INICIO'],
      dtype='object')

In [86]:
print(df_ctd_copy['VALOR'].unique().tolist())

[9.7306, 10.7823, 11.2579, 11.3263, 11.3199, 11.2614, 11.2521, 11.2078, 11.1349, 11.0797, 11.0477, 11.0185, 10.9861, 10.9602, 10.9563, 10.9547, 10.9449, 10.9368, 10.9312, 10.9282, 10.9263, 10.9245, 10.9216, 10.9185, 10.9159, 10.9126, 10.9082, 10.9047, 10.9035, 10.9037, 10.9032, 10.9019, 10.9006, 10.8998, 10.8993, 10.899, 10.8986, 10.8979, 10.8974, 10.8968, 10.8957, 10.8936, 10.8893, 10.8803, 10.8715, 10.8727, 10.881, 10.8877, 10.8987, 10.9003, 10.8992, 10.9005, 10.9026, 10.904, 10.9048, 10.9053, 10.9055, 10.9027, 10.8983, 10.8873, 10.8758, 10.8626, 10.8533, 10.851, 10.8516, 10.8511, 10.8506, 10.8504, 10.85, 10.8513, 10.853, 10.8538, 10.8546, 10.8536, 10.8535, 10.8543, 10.8548, 10.8555, 10.857, 10.8584, 10.8595, 10.8607, 10.8619, 10.8632, 10.8634, 10.8638, 10.8643, 10.8651, 10.8664, 10.8675, 10.8684, 10.8693, 10.8709, 10.8728, 10.874, 10.875, 10.876, 10.8772, 10.8797, 10.8836, 10.8869, 10.8887, 10.8895, 10.8902, 10.8907, 10.8914, 10.892, 10.8919, 10.8918, 10.8924, 10.893, 10.0576, 10.77

In [87]:
df_ctd_copy.to_csv('output/2024/AMBIENTALES_CTD_2024.csv' , index= False)

# Limpieza de FisicoQuimicos 2024

In [73]:
df_fisico = pd.read_excel(r'C:\Users\felipe.abarzua\Desktop\workspace\DATA_AMBIENTALES\Proyecto Seguimiento Ambiental\FISICO QUIMICOS_2012-2024\1_BD_FISICOQUIMICO_2024.xlsx')

In [ ]:
df_fisico['VALOR'].un

,Nombre del Proyecto:,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICULTURA EN CHILE Y SU EFECTO EN LOS ECOSISTEMAS DE EMPLAZAMIENTO 2024-2025,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Etapa Proyecto:,Objetivo 1. Actividad 2. Muestreos y Analisis ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jefe Proyecto,Johana Ojeda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Institucion:,INSTITUTO DE FOMENTO PESQUERO,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nombre Sector:,"Atacama (Caldera-Bahia Inglesa), Seno Reloncav...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Región:,"Región de Atacama,Región de Los Lagos, Región ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Johana Ojeda,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2024.Proy.SeguimientoAmbiental,Natales,9,"51°46'01,6''","72°54'05,0''",7.743333,...,12.5,0.52915,192.133333,21.413158,94.205825,16.523991,3.334888,2024-11-27 00:00:00,15:10:00,9
158,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Johana Ojeda,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2024.Proy.SeguimientoAmbiental,Natales,10,"52°03'57,8''","72°56'02,0''",7.28,...,8.5,0.141421,195.7,30.211918,34.419753,32.233468,4.886301,2024-11-27 00:00:00,20:20:00,10
159,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Johana Ojeda,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2024.Proy.SeguimientoAmbiental,Natales,11,"52°11'59,7''","72°57'03,6''",7.79,...,7.766667,0.057735,68,30.63968,109.928276,16.000306,9.156813,2024-11-28 00:00:00,13:30:00,11
160,ESTUDIO SEGUIMIENTO DESEMPEÑO AMBIENTAL ACUICU...,Objetivo 1. Actividad 2. Muestreos y Analisis ...,Johana Ojeda,INSTITUTO DE FOMENTO PESQUERO,Campaña_Otoño-Inv_2024.Proy.SeguimientoAmbiental,Natales,13,"52°13'10,4''","72°52'26,1''",8.093333,...,7.966667,0.11547,-39.8,23.906275,34.289202,4.395151,2.485062,2024-11-28 00:00:00,09:09:00,13


In [101]:
def limpieza_fisicoquimico(df):

    # Realizamos una copia del dataset
    df_fisico_copy = df.copy()

    # Eliminamos las filas innecesarias
    df_fisico_copy = df_fisico_copy.iloc[14: , :]

    # Dejamos la primera fila como columnas 
    df_fisico_copy.columns = df_fisico_copy.iloc[0]

    # Eliminamos la primera fila 
    df_fisico_copy = df_fisico_copy.iloc[1: , :]

    #Reiniciamos los indices
    df_fisico_copy = df_fisico_copy.reset_index(drop=True)

    #Seleccionamos las columnas
    df_fisico_copy = df_fisico_copy[['NOMB_PROY', 'ETAP_PROY', 'JEFE_PROY', 'INST_PROY', 'CAMPAÑA', 'ZONA',
       'ESTACION', 'Latitud S                ' , 'Longitud W ', 'pH' , 'Temperatura (°C)' , 'Redox EHNHE (mV)' , 'Sulfuros (µM)   ' , 'Materia orgánica total (%)',
        'Fecha',
       'Hora Muestreo' ]]

    # Agregamos la columna ID

    df_fisico_copy.insert(0 , 'ID' , df_fisico_copy.index + 1 )

    # Creamos una lista de columnas 

    lista_columnas = df_fisico_copy.columns.to_list()


    # Creamos una funcion para eliminar todo el contenido que esta en parentesis (*)

    def es_nan(x):
        return isinstance(x, float) and math.isnan(x)


    def quitar_parentesis_contenido(lista):
    
        resultado = []
        patron = re.compile(r'\([^)]*\)')  # todo lo que esté entre paréntesis (no anida)
        for c in lista:
            if es_nan(c) or c is None:
                resultado.append(c)
            elif isinstance(c, str):
                s = patron.sub('', c)                 # quita ( ... )
                s = re.sub(r'\s{2,}', ' ', s).strip() # colapsa espacios y recorta
                resultado.append(s)
            else:
                resultado.append(c)
        return resultado

    # Aplicamos la funcion para limpiar las columnas 
    lista_columnas = quitar_parentesis_contenido(lista_columnas)

    # Cambiamos las columnas del dataset

    df_fisico_copy.columns = lista_columnas

    # Creamos las columnas fijas y columnas variables

    indices_fijos = [0 , 1 , 2 ,3 ,4 ,5 ,6 ,7 , 8 , 9 ,  15 , 16]

    columnas_fijas = [lista_columnas[i] for i in indices_fijos]

    columnas_variables = [col for i, col in enumerate(lista_columnas) if i not in indices_fijos]

    
    # Ahora realizamos un melt para convertir las columnas variables en filas

    df_fisico_copy = pd.melt(df_fisico_copy , id_vars= columnas_fijas , value_vars= columnas_variables , var_name= 'VARIABLE' , value_name='VALOR' )

    # Convertimos la columna de VARIABLE a mayuscula

    df_fisico_copy['VARIABLE'] = df_fisico_copy['VARIABLE'].str.upper()

    #Corregimos los variables 's/d'
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 'SIN DATO', np.nan , df_fisico_copy['VALOR'] )
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 'sd', np.nan , df_fisico_copy['VALOR'] )
    df_fisico_copy['VALOR'] = np.where(df_fisico_copy['VALOR'] == 's/d', np.nan , df_fisico_copy['VALOR'] )
    
    #Eliminamos las filas con VALOR no nulo 

    df_fisico_copy = df_fisico_copy[df_fisico_copy['VALOR'].notnull()]

    #Reiniciamos los indices    
    df_fisico_copy = df_fisico_copy.reset_index(drop = True)

    # Cambiamos el tipo de dato de la columna Fecha 

    df_fisico_copy['Fecha'] = pd.to_datetime(df_fisico_copy['Fecha'], errors='coerce')

    #Creamos una columna para unir la Fecha y Hora Muestreo

    # Convertimos las Hora_Muestreo que son invalidos a 00:00:00
    df_fisico_copy['Hora Muestreo'] = np.where(df_fisico_copy['Hora Muestreo'] == 'sd' , datetime.time(0,0) , df_fisico_copy['Hora Muestreo'])

    # Convertimos los errores de la Hora Muestreo '14.40' Y '   10:50'

    df_fisico_copy['Hora Muestreo'] = np.where(df_fisico_copy['Hora Muestreo'] == '   10:50' , datetime.time(10,50) , df_fisico_copy['Hora Muestreo'])
    df_fisico_copy['Hora Muestreo'] = np.where(df_fisico_copy['Hora Muestreo'] == '14.40' , datetime.time(14,40) , df_fisico_copy['Hora Muestreo'])
    


    #Creamos la columna llamada FECHA_MUESTREO que combine la fecha y la hora de muestreo
    df_fisico_copy['FECHA_MUESTREO'] = df_fisico_copy['Fecha'].dt.strftime('%d/%m/%Y') + " " + df_fisico_copy['Hora Muestreo'].astype(str)

    # Convertimos las variables 
    df_fisico_copy['VARIABLE'] = np.where(df_fisico_copy['VARIABLE'] == 'MATERIA ORGÁNICA TOTAL' , 'MATERIA ORGANICA' , df_fisico_copy['VARIABLE'])


    def normalizar(s):
            if pd.isna(s):
                return s
            # quitar tildes, bajar a minúsculas y recortar espacios
            s = ''.join(c for c in unicodedata.normalize('NFD', str(s).strip().lower())
                        if unicodedata.category(c) != 'Mn')
            return s

    # Agregamos la columna de COD_REGION
    
    # Diccionario en números romanos
        
    mapa_romano = {
            # Los Lagos (X)
            'reloncavi': 'X',
            'estuario': 'X',
            'calbuco': 'X',
            'maullin': 'X',
            'ancud': 'X',
            'chacao': 'X',
            'butachauques': 'X',
            'chaulinec': 'X',
            'tenaun': 'X',
            'chiloe central': 'X',
            'desertores': 'X',
            'quellon': 'X',
            'hornopiren': 'X',
            'ayacara': 'X',
            'chaiten': 'X',

            # Aysén (XI)
            'guaitecas': 'XI',
            'moraleda': 'XI',
            'puyuhuapi': 'XI',
            'aysen': 'XI',

            # Coquimbo (IV)
            'coquimbo': 'IV',
            'tongoy': 'IV',

            # Atacama (III)
            'atacama': 'III',

            # Magallanes (Natales) – XII
            'natales': 'XII',

            # Biobío (VIII)
            'bio-bio': 'VIII',

            # Los Ríos (XIV)
            'valdivia': 'XIV',
        }

    # Aplicamos la funcion para obtener el COD_REGION
    df_fisico_copy['COD_REGION'] = df_fisico_copy['ZONA'].apply(normalizar).map(mapa_romano)

    #Aplicamos una funcion para eliminar las ' y ° de LATITUD Y LONGITUD 

    df_fisico_copy['Longitud W'] = df_fisico_copy['Longitud W'].str.replace(r"[°'\"]", "", regex=True).str.rstrip().str.replace(',' , '.')
    df_fisico_copy['Latitud S'] = df_fisico_copy['Latitud S'].str.replace(r"[°'\"]", "", regex=True).str.rstrip().str.replace(',' , '.')

    #Renombramos los dataframes

    df_fisico_copy = df_fisico_copy.rename(columns={'Longitud W' : 'LONGITUD' , 'Latitud S' : 'LATITUD'})

  
    return df_fisico_copy , lista_columnas 

In [103]:
df_fisico_copy , lista_columnas = limpieza_fisicoquimico(df_fisico)

In [106]:
df_fisico_copy.to_csv('output/2024/AMBIENTALES_FISICOQUIMICO_2024.csv' , index= False)

In [91]:
df_fisico_copy.columns

Index(['ID', 'NOMB_PROY', 'ETAP_PROY', 'JEFE_PROY', 'INST_PROY', 'CAMPAÑA',
       'ZONA', 'ESTACION', 'LATITUD', 'LONGITUD', 'Fecha', 'Hora Muestreo',
       'VARIABLE', 'VALOR', 'FECHA_MUESTREO', 'COD_REGION'],
      dtype='object')

In [105]:
df_fisico_copy['FECHA_MUESTREO'].unique()

array(['20/05/2024 09:35:00', '20/05/2024 09:16:00',
       '19/05/2024 16:20:00', '19/05/2024 15:50:00',
       '19/05/2024 15:00:00', '19/05/2024 15:30:00',
       '19/05/2024 14:15:00', '19/05/2024 11:00:00',
       '19/05/2024 09:45:00', '19/05/2024 09:00:00',
       '21/05/2024 15:20:00', '21/05/2024 16:20:00',
       '22/05/2024 09:00:00', '23/05/2024 08:50:00',
       '18/05/2024 08:55:00', '18/05/2024 10:05:00',
       '18/05/2024 11:00:00', '18/05/2024 12:00:00',
       '18/05/2024 13:55:00', '21/05/2024 14:25:00',
       '23/05/2024 08:20:00', '22/05/2024 16:10:00',
       '23/05/2024 10:20:00', '23/05/2024 11:30:00',
       '22/05/2024 15:30:00', '22/05/2024 12:00:00',
       '22/05/2024 14:30:00', '22/05/2024 14:50:00',
       '24/05/2024 12:00:00', '24/05/2024 11:30:00',
       '24/05/2024 11:40:00', '24/05/2024 11:10:00',
       '25/05/2024 10:45:00', '25/05/2024 11:20:00',
       '25/05/2024 12:40:00', '25/05/2024 10:00:00',
       '25/05/2024 08:45:00', '25/05/2024 08:2